In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier, Pool

In [ ]:
df_train = pd.read_csv("/kaggle/input/titanic/train.csv")
df_test = pd.read_csv("/kaggle/input/titanic/test.csv")
df_sample = pd.read_csv("/kaggle/input/titanic/gender_submission.csv")

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
print(df_train.isnull().sum())
print("\n")
print(df_test.isnull().sum())

In [ ]:
df_train.drop(columns = ["PassengerId", "Name", "Cabin", "Ticket"], inplace = True)
df_test.drop(columns = ["Name", "Cabin", "Ticket"], inplace = True)

In [ ]:
print(df_train.columns.tolist())
print("\n")
print(df_test.columns.tolist())

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
print(df_train.isnull().sum())
print("\n")
print(df_test.isnull().sum())

In [ ]:
df_test_full = df_test.copy()

In [ ]:
X = df_train.drop(columns = "Survived")
Y = df_train["Survived"]

In [ ]:
categorical_cols = [
    "Pclass", "Sex", "SibSp", "Parch", "Embarked"
]

continuous_col = [
    "Age", "Fare"
]

In [ ]:
for col in continuous_col:
    lower, upper  = X[col].quantile(0.01), X[col].quantile(0.99)
    X[col] = X[col].clip(lower, upper)
    df_test[col] = df_test[col].clip(lower, upper)

In [ ]:
# ---------------------
# Ensure categorical columns are strings
# ---------------------
for col in categorical_cols:
    X[col] = X[col].astype(str)
    df_test[col] = df_test[col].astype(str)

df_test = df_test[X.columns]
cat_features = [X.columns.get_loc(col) for col in categorical_cols]

# ---------------------
# Model Parameters (GPU)
# ---------------------
params = {
    "iterations": 1000,
    "depth": 5,
    "learning_rate": 0.22775461488,
    "l2_leaf_reg": 7.46314929,
    "bagging_temperature": 0.0350283198,
    "border_count": 252,
    "random_strength": 1.59045421e-05,
    "eval_metric": "Accuracy",   # ✅ Changed from AUC to Accuracy
    "loss_function": "Logloss",
    "random_seed": 42,
    "verbose": False,
    "task_type": "GPU",          # ⚡ Enable GPU
    "devices": "0"               # Use first GPU device
}

cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)

acc_scores = []
all_preds = []
oof_preds = np.zeros(len(Y))

# ---------------------
# Cross-validation with GPU
# ---------------------
for fold, (train_idx, val_idx) in enumerate(cv.split(X, Y), 1):
    print(f"\n🚀 Training Fold {fold} on GPU...")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    Y_train, Y_val = Y.iloc[train_idx], Y.iloc[val_idx]

    train_pool = Pool(X_train, Y_train, cat_features=cat_features)
    val_pool = Pool(X_val, Y_val, cat_features=cat_features)

    model = CatBoostClassifier(**params, scale_pos_weight=len(Y[Y == 0]) / len(Y[Y == 1]))
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    # Predict labels for validation
    Y_pred = model.predict(val_pool)
    acc = accuracy_score(Y_val, Y_pred)
    acc_scores.append(acc)
    oof_preds[val_idx] = Y_pred

    # Predict on test data (GPU)
    test_pool = Pool(df_test, cat_features=cat_features)
    test_preds = model.predict(test_pool)
    all_preds.append(test_preds)

    print(f"Fold {fold} Accuracy: {acc:.5f}")

# ---------------------
# Results
# ---------------------
print(f"\n✅ Mean Accuracy across folds: {np.mean(acc_scores):.5f}")

In [ ]:
df_sample

In [ ]:
preds = np.mean(all_preds, axis = 0)
submission = pd.DataFrame({
    "PassengerId": df_test_full["PassengerId"],
    "Survived": preds
})
submission.to_csv("submission.csv", index = False)
submission.to_csv("/kaggle/working/submission.csv", index = False)